# RASCAR V27: exact replay
Frozen score components, not end-to-end training. Public AP unknown; best confirmed V22 = 0.95610.
Cross-day primary OOF = 0.9848634937162036. Candidate differs from control only for hand #266.

In [ ]:
import base64, io
from pathlib import Path
import pandas as pd, numpy as np
from sklearn.metrics import average_precision_score
PAYLOAD = {'replay_inputs/oof.csv': '...', 'replay_inputs/test.csv': '...'}
for split, encoded in PAYLOAD.items():
    d = pd.read_csv(io.BytesIO(base64.b64decode(encoded)))
    for mode in ['control', 'candidate']:
        score = (d.assembled + d.merged_delta + (d.weak_hand_delta if mode == 'candidate' else 0)).clip(0, 1)
        out = pd.DataFrame({'id': d.id, 'score': score})
        if split.endswith('test.csv'):
            assert len(out) == 362 and out.id.is_unique and np.isfinite(score).all()
            out.to_csv(f'submission_v27_{mode}.csv', index=False)
            print(mode, 'saved', len(out))
        else:
            print(mode, 'OOF AP', average_precision_score(d.error, score))
